# Install Dependecies

In [2]:
%%capture
%pip install -q "nltk>=3.9,<4" "spacy>=3.8,<4" "transformers>=5,<6"
%pip install matplotlib
!python -m spacy download en_core_web_sm
!python -m spacy download fr_core_news_sm
%pip install ipynbname
%pip install datasets
%pip install ipywidgets

In [3]:
%%capture
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu130
%pip install ipykernel

# Imports

In [4]:
import ipynbname
import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
import re
import time
from collections import Counter
from datasets import load_dataset
from pathlib import Path
from tqdm.std import tqdm
from transformers import AutoTokenizer

# Import datasets

In [6]:
dataset = load_dataset("coastalcph/tydi_xor_rc")
df_train = dataset["train"].to_pandas()
df_val = dataset["validation"].to_pandas()

# Preprocessing

In [7]:
LANGUAGES = ["ar", "ko", "te"]

In [8]:
# select languages
df_train = df_train[df_train["lang"].isin(LANGUAGES)].copy()
df_val = df_val[df_val["lang"].isin(LANGUAGES)].copy()

# Tokeniser

In [25]:
xlm_tokeniser = AutoTokenizer.from_pretrained("xlm-roberta-base")

# 3 Week 37: Structured Span Prediction
* Convert the character-level answer offsets into BIO labels over context tokens. 
* Add automatic checks for at least the following cases: 
  - an answer at character 0, 
  - a multi-token answer, 
  - punctuation adjacent to an answer and an unanswerable example. 
* Document how subword pieces are handled if applicable. 
* Implement one question-conditioned sequence labeller for the group: the representation of the question must influence the predicted label for every context token. 
* Compare it with a simple span baseline. An empty-output baseline is sufficient. If you use lexical overlap, select context tokens using only overlap with the question (optionally after fixed preprocessing or translation), convert the best contiguous run to a span and never use gold answer text or offsets. The correct output for an unanswerable question is an empty span. Evaluate and analyse the models according to Section 1.


### 1. Convert the character-level answer offsets into BIO labels over context tokens

#### 1.1 Pin down the character span
The gold span is `answer_start` to `answer_start + len(answer)`, end exclusive. Project 1 already verified that slicing the context with these gives back the answer text. Also confirm what unanswerable rows look like, for example `answer_start == -1` and an empty answer, so they can be routed to the all-O case explicitly.

In [26]:
def span_interval(df):
    df = df.copy()
    df["char_start"] = -1
    df["char_end"] = -1
    mask = df["answerable"]  # only apply to rows where answerable == True
    df.loc[mask, "char_start"] = df.loc[mask, "answer_start"]
    df.loc[mask, "char_end"] =  df.loc[mask, "answer_start"] + df.loc[mask, "answer"].str.len()
    return df

In [27]:
df_train = span_interval(df_train)
df_val = span_interval(df_val)
df_train.head(1)

,question,context,lang,answerable,answer_start,answer,answer_inlang,char_start,char_end
4792,30년 전쟁의 승자는 누구인가?,The conflict between France and Spain continue...,ko,True,21,France,None,21,27


#### 1.2 Tokenise with offsets
Call the XLM-R fast tokeniser with `return_offsets_mapping=True`. Each token then comes with its character start and end in the original string. Decide here whether to tokenise the context alone or the question and context as a pair. The pair matches what the labeller will consume later, and `sequence_ids()` tells you which tokens belong to the context. Offsets for the context tokens stay relative to the context string either way.

In [31]:
row = df_train.iloc[0]
question, context = row["question"], row["context"]
print(row["answer"], row["char_start"], row["char_end"])

France 21 27


In [45]:
encoding = xlm_tokeniser(question, context, return_offsets_mapping=True, truncation=True)
encoding.keys()

KeysView({'input_ids': [0, 496, 2680, 134293, 367, 67134, 48894, 116932, 114150, 32, 2, 2, 581, 79612, 17721, 9942, 136, 84740, 136475, 23, 128342, 43265, 24189, 611, 12975, 4, 678, 70, 106457, 1363, 17721, 6626, 95934, 38529, 7, 136, 6626, 168655, 27759, 7, 4, 1632, 35509, 23, 5755, 4, 1379, 70, 6226, 111, 84740, 136, 70, 3789, 23, 908, 5128, 53, 1298, 4, 1379, 70, 6, 167618, 111, 9942, 5, 360, 611, 12744, 70, 92265, 207048, 10670, 85018, 71, 47, 128342, 43265, 25, 7, 43396, 90, 127067, 111, 70, 166259, 1151, 90, 4, 1284, 34658, 6226, 111, 56709, 7, 7, 62339, 4, 2685, 1272, 105207, 47, 70, 1144, 592, 111, 70, 4804, 67530, 111, 70, 166259, 1151, 90, 23, 611, 12975, 4, 3129, 77681, 134620, 70, 1631, 17721, 9942, 136, 84740, 4, 678, 70, 2878, 1363, 111, 3332, 4935, 128342, 43265, 17721, 15044, 20244, 50964, 5, 581, 82528, 21325, 13, 9624, 1290, 2320, 5550, 134620, 678, 70, 4804, 67530, 111, 230991, 23, 611, 16028, 4, 450, 121011, 297, 70, 42698, 46799, 87, 21547, 66, 32528, 5, 2], 'atten

In [63]:
# question tokens: sequence_id = 0
# context tokens: sequence_id = 1
# special chars: sequence_id = None
tokens = xlm_tokeniser.convert_ids_to_tokens(encoding["input_ids"])
for token, offset, sequence_id in zip(tokens, encoding["offset_mapping"], encoding.sequence_ids()):
    print(f"token: {token}, offset: {offset}, sequence_id: {sequence_id}")

token: <s>, offset: (0, 0), sequence_id: None
token: ▁30, offset: (0, 2), sequence_id: 0
token: 년, offset: (2, 3), sequence_id: 0
token: ▁전쟁, offset: (4, 6), sequence_id: 0
token: 의, offset: (6, 7), sequence_id: 0
token: ▁승, offset: (8, 9), sequence_id: 0
token: 자는, offset: (9, 11), sequence_id: 0
token: ▁누구, offset: (12, 14), sequence_id: 0
token: 인가, offset: (14, 16), sequence_id: 0
token: ?, offset: (16, 17), sequence_id: 0
token: </s>, offset: (0, 0), sequence_id: None
token: </s>, offset: (0, 0), sequence_id: None
token: ▁The, offset: (0, 3), sequence_id: 1
token: ▁conflict, offset: (4, 12), sequence_id: 1
token: ▁between, offset: (13, 20), sequence_id: 1
token: ▁France, offset: (21, 27), sequence_id: 1
token: ▁and, offset: (28, 31), sequence_id: 1
token: ▁Spain, offset: (32, 37), sequence_id: 1
token: ▁continued, offset: (38, 47), sequence_id: 1
token: ▁in, offset: (48, 50), sequence_id: 1
token: ▁Catal, offset: (51, 56), sequence_id: 1
token: onia, offset: (56, 60), sequence_id:

In [47]:
for i, (token, (start, end), seq_id) in enumerate(zip(tokens, encoding["offset_mapping"], encoding.sequence_ids())):
    if seq_id == 1 and start < row["char_end"] and end > row["char_start"]:
        print(f"idx: {i}: token: {token}, (start: {start}, end: {end})")

idx: 15: token: ▁France, (start: 21, end: 27)


#### 1.3 Inspect how the offsets behave

In [ ]:
# B: token that begins a span of interest
# I: token inside a span
# O: token outside any span
BIO_LABELS = {"B": 1, "I": 2, "O": 0}
IGNORE = -100

In [70]:
def bio_labels(row, tokeniser=xlm_tokeniser):
    encoding = tokeniser(row["question"], row["context"], return_offsets_mapping=True, truncation=True)
    start, end = row["char_start"], row["char_end"]
    sequence_id = encoding.sequence_ids()

    labels = []
    seen_answer = False
    for (token_start, token_end), sequence_id in zip(encoding["offset_mapping"], sequence_id):
        if sequence_id != 1:
            labels.append(IGNORE)
            continue
        if start == -1:
            labels.append(BIO_LABELS["O"])
            continue
        overlaps = token_start < end and token_end > start and token_end > token_start
        if not overlaps:
            labels.append(BIO_LABELS["O"])
        elif not seen_answer:
            labels.append(BIO_LABELS["B"])
            seen_answer = True
        else:
            labels.append(BIO_LABELS["I"])
    assert len(labels) == len(encoding["input_ids"])
    return encoding, labels

In [71]:
ID2LABEL = {v: k for k, v in BIO_LABELS.items()}  # {1: "B", 2: "I", 0: "O"}
ID2LABEL[IGNORE] = "-"                            # special and question tokens

def show_row_compact(row, title):
    encoding, labels = bio_labels(row)
    tags = [ID2LABEL[l] for l in labels]
    n_ignore = sum(t == "-" for t in tags)
    n_o = sum(t == "O" for t in tags)
    n_b = sum(t == "B" for t in tags)
    n_i = sum(t == "I" for t in tags)
    print(f"=== {title} ===")
    print("question:", row["question"])
    print("answer:  ", row["answer"] if row["char_start"] != -1 else "<unanswerable>")
    print("tags:    ", " ".join(tags))
    print(f"counts:   IGNORE={n_ignore}  O={n_o}  B={n_b}  I={n_i}")
    print()

def first_row_where(cond):
    for _, r in df_train.iterrows():
        encoding, labels = bio_labels(r)
        if cond(labels):
            return r

single = first_row_where(lambda l: l.count(BIO_LABELS["B"]) == 1 and BIO_LABELS["I"] not in l)
multi  = first_row_where(lambda l: BIO_LABELS["I"] in l)
unans  = first_row_where(lambda l: BIO_LABELS["B"] not in l)

show_row_compact(single, "single-token answer: B only")
show_row_compact(multi,  "multi-token answer: B then I")
show_row_compact(unans,  "unanswerable: all O")

=== single-token answer: B only ===
question: 30년 전쟁의 승자는 누구인가?
answer:   France
tags:     - - - - - - - - - - - - O O O B O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O -
counts:   IGNORE=13  O=160  B=1  I=0

=== multi-token answer: B then I ===
question: 엑스선은 누가 발견하였는가?
answer:   Wilhelm Röntgen
tags:     - - - - - - - - - - - - - O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O B I I I O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O O 

### 1.4 Define the token labelling rule
Done as code, missing the prose. Move `BIO_LABELS`, `IGNORE`, `bio_labels` and the three demo prints here, and add the markdown cell describing the rule and subword handling.

### 1.5 Label both splits
in progress. One cell per split, storing `input_ids`, `attention_mask`, `offset_mapping` and labels as columns, followed by the length assert.

### 1.6 Count truncation losses
Answerable rows with no B in their labels, per split and per language.

### 1.7 Decode labels back to a span
The inverse function, from labels and offsets to `context[start:end]`, with the exact match rate over all answerable rows.